# 36 · Skill 触发 sub-agent + meta-Skill 组合

> **学习目标**：演示 Claude Code 里的两类**高级 Skill 用法** —— (a) Skill 步骤里**让 Claude Code spawn sub-agent** 处理耗 context 的子任务；(b) meta-Skill **内部编排多个子 Skill** 形成个人工作流目录。
>
> **预备**：33/34/35 跑过；02-Agent 30/31 跑过。
>
> **为什么重要**：单个 Skill 是**原子能力**，组合 Skill 才是**工作流**。组合得对 → 你的「个人 / 项目工作流目录」就建起来了。

In [ ]:
import os, shutil, json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Callable
SBX = Path('./_skill_sandbox').resolve()
if SBX.exists(): shutil.rmtree(SBX)
SBX.mkdir()
HOME_SKILLS = SBX / 'home' / '.claude' / 'skills'
HOME_SKILLS.mkdir(parents=True)
print('沙箱就绪')

## 1. 模式 A：Skill 触发 sub-agent

**Why**：某些任务（搜代码、读大量文件、跑长 search）会**污染主对话的 context**。
**做法**：Skill 的 Steps 写「**用 Agent 工具 spawn 一个子 agent**，限定它的任务范围，结果回主对话时**只回摘要**」。

下面做一个 `/audit-rag` Skill：审计 [rag_project/](../../../../rag_project/) 当前 RAG 配置 + 最近改动，返回**一段精炼结论**，而不是把整个 diff 灌进主对话。

In [ ]:
audit_dir = HOME_SKILLS / 'audit-rag'
audit_dir.mkdir()
(audit_dir / 'SKILL.md').write_text('''---
name: audit-rag
description: When the user asks to audit or review a RAG project configuration. Trigger: "audit rag", "review my RAG", "rag 评估", "看看 rag 配置". Do NOT trigger for: building a new RAG from scratch, or running a single query test.
---

# When to use
User wants a **concise review** of the RAG project's config + recent changes. They want signal, not raw diff.

# When NOT to use
- User is building a brand new RAG (use the RAG chapter Stage 1 instead)
- User wants to run a single query test (just use `python -m rag.cli query`)

# Steps
1. Use the **Agent tool** to spawn a sub-agent (type: `general-purpose`)
2. Give the sub-agent this exact task:
   > Read `config/default.yaml`, recent commits in the last 7 days (git log -7), and `eval/eval_v*.jsonl` if exists.
   > Identify the most important 3-5 issues (e.g., wrong embedding model, missing reranker, oversized chunks, stale eval set). For each, propose 1 specific fix.
   > **Do NOT** return the full content. Return a summary ≤ 400 words + bullet list of issues.
3. When the sub-agent returns, **you (main) summarize further** to ≤ 150 words + numbered list
4. Add a final line: "Want me to fix any of these? Tell me the number."

# Why use a sub-agent
- Sub-agent reads files / git log / eval JSONLs in **its own context**
- Main conversation stays clean
- Sub-agent has its own budget; can do 20+ Read calls without polluting main

# Example
User: "audit my RAG"
→ Sub-agent reads 3-5 files, returns summary
→ Main returns 150-word summary + 3 issues + ask for which to fix
''', encoding='utf-8')
print('已建 Skill: audit-rag/')

## 2. 模拟「Skill → sub-agent → main」三段执行

**我们**用一个简化的 simulator 演示：
- `MockSubAgent.run(task)` —— 在自己 context 里跑（不污染主）
- 主对话拿到 `summary`
- 演示**为什么 sub-agent 是 Claude Code 的关键设计**

In [ ]:
@dataclass
class SubAgentResult:
    output: str
    internal_steps: int
    internal_chars: int
    main_visible_chars: int

class MockSubAgent:
    def run(self, task: str, verbose: bool = False) -> SubAgentResult:
        steps = 8
        files_read = {
            'config/default.yaml': 2000,
            'git log -7': 800,
            'eval/eval_v1.jsonl': 1500,
        }
        internal_chars = sum(files_read.values())
        summary = self._fake_summary(task, files_read)
        if verbose:
            print(f'    [sub-agent 内部 steps={steps}, 读了 {internal_chars} 字符]')
        return SubAgentResult(
            output=summary,
            internal_steps=steps,
            internal_chars=internal_chars,
            main_visible_chars=len(summary),
        )

    @staticmethod
    def _fake_summary(task: str, files: dict) -> str:
        return (
            '【RAG 审计报告】\n'
            '1. embedding 默认 ollama/nomic-embed-text，中文召回效果不如 bge-m3，建议加 rerank\n'
            '2. chunk_size=500, overlap=80 对技术手册过短，PDF 多表格丢失\n'
            '3. 检索只用 cosine dense，无 hybrid；query 长时召回掉\n'
            '4. eval set 仅 5 题，无回归 CI\n'
            '5. recent commits 集中在 cli.py，loader / chunker / vectorstore 几月没动\n'
        )

sub = MockSubAgent()
result = sub.run('audit rag_project', verbose=True)
print(f'\nmain 实际看到（{result.main_visible_chars} 字符 vs sub 内部 {result.internal_chars} 字符）:')
print(result.output)
print(f'\n→ 节省 {result.internal_chars - result.main_visible_chars} 字符不进入主对话（节省 {(1 - result.main_visible_chars/result.internal_chars)*100:.0f}%）')

## 3. 模式 B：meta-Skill 编排多个子 Skill

**核心思想**：写一个 Skill A，它的 Steps 调 Skill B、C、D 解决子问题，**A 是工作流，B/C/D 是原子能力**。

实战例子：`/study-topic` 帮你深入研究一个话题 → 自动 spawn 3 个子 Skill（/search-web / /summarize / /write-notes）。

In [ ]:
import yaml
def make_skill(name: str, description: str, body: str):
    d = HOME_SKILLS / name
    d.mkdir(exist_ok=True)
    (d / 'SKILL.md').write_text(f'''---
name: {name}
description: {description}
---

{body}
''', encoding='utf-8')
    return d

make_skill('search-web',
           'When the user wants to search the web for a topic. Trigger: "search the web", "google it", "搜一下", "查一下".',
           '''# Steps
1. Use WebSearch tool with the topic
2. Return top 5 results' summary
''')
make_skill('summarize',
           'When the user wants to summarize a long text. Trigger: "summarize", "总结", "TLDR", "give me the gist".',
           '''# Steps
1. Receive the text
2. Extract 3-5 key points
3. Return bullet list
''')
make_skill('write-notes',
           'When the user wants to write notes / a memo to disk. Trigger: "save this", "write notes", "记下来", "备忘一下".',
           '''# Steps
1. Receive content
2. Use Write tool to create notes/<topic>.md
3. Return file path
''')
print('已建 3 个原子 Skill: search-web, summarize, write-notes')

In [ ]:
make_skill('study-topic',
           'When the user wants a deep-dive study of a topic. Trigger: "study", "深入了解", "帮我研究", "study topic". Do NOT trigger for: simple web search or single-sentence answers.',
           '''# When to use
User wants a **structured** deep-dive: search + summarize + save notes.

# When NOT to use
- User just wants a quick web search (use /search-web directly)
- User just wants a single answer (respond directly)

# Steps
1. Use `/search-web` to get top 5 results for the topic
2. Use `/summarize` on the search results (pass the text inline)
3. Use `/write-notes` to save the summary to `notes/<topic>.md`
4. Tell the user: "研究完成。已保存到 notes/<topic>.md。要继续深挖某个点吗？"

# Example
User: "study RAG evaluation metrics"
→ search-web "RAG evaluation metrics 2024"
→ summarize the results
→ write-notes "rag-eval-metrics.md" with the summary
''')
print('已建 meta-Skill: study-topic')

In [ ]:
def run_study_topic(topic: str, verbose: bool = True) -> dict:
    trace = []
    search_result = (
        f'(5 个 web 搜索结果关于 "{topic}":\n'
        f'  1. blog1: ...\n'
        f'  2. paper: ...\n'
        f'  3. github: ...\n'
        f'  4. doc: ...\n'
        f'  5. news: ...)'
    )
    trace.append(('search-web', search_result))
    if verbose: print(f'  [1] /search-web → {len(search_result)} 字符')
    summary = (
        f'【{topic} 总结】\n'
        f'- 关键点 1\n'
        f'- 关键点 2\n'
        f'- 关键点 3'
    )
    trace.append(('summarize', summary))
    if verbose: print(f'  [2] /summarize → {len(summary)} 字符')
    note_path = SBX / 'notes' / f'{topic.replace(" ", "-")}.md'
    note_path.parent.mkdir(parents=True, exist_ok=True)
    note_path.write_text(summary, encoding='utf-8')
    trace.append(('write-notes', str(note_path)))
    if verbose: print(f'  [3] /write-notes → {note_path}')
    return {'trace': trace, 'final': f'研究完成。已保存到 {note_path}'}

result = run_study_topic('RAG evaluation metrics')
print(f'\nFinal Answer: {result["final"]}')
print()
print('生成的笔记:')
print('-' * 40)
print((SBX / 'notes' / 'RAG-evaluation-metrics.md').read_text(encoding='utf-8'))

## 4. 两种模式对比

| 维度 | A: Skill → sub-agent | B: meta-Skill → 子 Skills |
|------|----------------------|----------------------------|
| **目的** | 隔离 context | 组合工作流 |
| **运行时** | sub-agent 自己跑工具 | main 串行触发各 Skill |
| **返回值** | 短摘要（200-500 字符） | 各 Skill 自己的结果 |
| **对 main 干扰** | 极小（context 隔离） | 中（每 Skill 都用 main 的 context） |
| **适用** | 重型读文件 / 搜索任务 | 串行流水线 |
| **反模式** | 啥都包进 sub-agent | meta-Skill 调 10+ 个子 Skill |

**经验**：先 A、再 B。**A 是「这步太重」问题；B 是「这步要拆」问题**。

In [ ]:
shutil.rmtree(SBX, ignore_errors=True)
print('沙箱清理')

## 深入思考

1. **meta-Skill 调子 Skill 怎么传结果？**
   - 通过**对话历史**。main 自己读 Skill B 的结果，**拼接后**再喂给 Skill C。**没有「Skill 间直接传递」的机制**。这是有意的设计（让 main 始终「知情」）。
2. **sub-agent 失败怎么办？**
   - main 应检查 `result.isError`，给用户兜底回答。**永远不要**假设 sub-agent 100% 成功。
3. **meta-Skill 嵌套层数？**
   - 经验：≤ 3 层。**太深 → 调试时找问题链路难**。
4. **能不能一个 Skill 在 description 里 trigger 另一个 Skill？**
   - 间接：通过 Steps 写「先 `/skill-x`」让 main 串行触发。**没有「Skill 自动 trigger Skill」机制**。
5. **sub-agent 适合跑 RAG / 搜索 / 跑测试 / 改文件吗？**
   - 前 3 个：适合。**第 4 个（改文件）** —— sub-agent 不该有写权限，**应该用 AskUserQuestion 把方案给 main 让用户确认**。

**改一改**：
- 让 `study-topic` 的 Step 2 (summarize) 也 spawn sub-agent（演示「meta-Skill + sub-agent 组合」）
- 写一个你的工作流目录的 meta-Skill（3-4 个原子 Skill 组合）

## 自检 ✅

- [ ] 默写「Skill 触发 sub-agent」的好处 3 个
- [ ] 解释「为什么 sub-agent 不该有写权限」
- [ ] 解释「meta-Skill 调子 Skill 是怎么传结果的」
- [ ] 默写「sub-agent 失败兜底」3 步
- [ ] 给一个工作流，能判断该用 sub-agent 还是 meta-Skill

## 下一步

→ [`37_skill_hooks_permissions.ipynb`](37_skill_hooks_permissions.ipynb)